# Week 08 · Friday — Transfer Learning + ME1 Preparation
**PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar**

Dataset: `medical_imaging_meta.csv` (520 labeled rows, 5 conditions, 30 unlabeled images)  
Reference: Chest X-Ray Pneumonia — Kaggle / NIH Chest X-ray

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, roc_auc_score
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Sub-step 1 — Data Loading and Characterisation (Easy / Required)

In [ ]:
CSV_PATH = Path('medical_imaging_meta.csv')
IMAGE_ROOT = Path('images')  # adjust to your local image folder

def load_metadata(csv_path: Path) -> pd.DataFrame:
    """Load the metadata CSV and return a clean DataFrame."""
    if not csv_path.exists():
        raise FileNotFoundError(
            f'CSV not found at {csv_path}. '
            'Download from LMS and place it alongside this notebook.'
        )
    df = pd.read_csv(csv_path)
    print(f'Loaded {len(df)} rows, {df.shape[1]} columns')
    return df

df = load_metadata(CSV_PATH)
df.head()

In [ ]:
def characterise_dataset(df: pd.DataFrame, label_col: str = 'condition',
                          site_col: str = 'hospital_site',
                          quality_col: str = 'image_quality') -> None:
    """
    Examine label distribution, minority classes, and subgroup differences
    across hospital site and image quality.
    """
    print('=== Label distribution ===')
    label_counts = df[label_col].value_counts()
    print(label_counts.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # class counts
    sns.barplot(x=label_counts.index, y=label_counts.values, ax=axes[0], palette='muted')
    axes[0].set_title('Class distribution (labeled images)')
    axes[0].set_ylabel('Count')
    axes[0].set_xlabel('Condition')
    for bar, count in zip(axes[0].patches, label_counts.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                     str(count), ha='center', va='bottom', fontsize=9)

    # site breakdown
    if site_col in df.columns:
        site_label = df.groupby([site_col, label_col]).size().unstack(fill_value=0)
        site_label.plot(kind='bar', ax=axes[1], colormap='tab10')
        axes[1].set_title('Condition count by hospital site')
        axes[1].set_xlabel('Hospital site')
        axes[1].set_ylabel('Count')
        axes[1].legend(loc='upper right', fontsize=7)
    else:
        axes[1].set_visible(False)

    plt.tight_layout()
    plt.savefig('plots/label_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    # minority class check
    minority_threshold = label_counts.max() * 0.3
    minority_classes = label_counts[label_counts < minority_threshold].index.tolist()
    print(f'\nMinority classes (< 30% of majority): {minority_classes}')

    # image quality subgroup if available
    if quality_col in df.columns:
        print('\n=== Image quality breakdown ===')
        print(df.groupby([quality_col, label_col]).size().unstack(fill_value=0))

    print(
        '\nFinding: The dataset is class-imbalanced. Minority classes correspond '
        'to conditions that are clinically critical (e.g., effusion, pneumothorax). '
        'A model optimised for overall accuracy will systematically underperform on '
        'precisely the cases Dr. Rao cannot afford to miss. '
        'Hospital-site variation also introduces potential confounding — '
        'a model may learn scanner artefacts rather than pathology.'
    )
    return minority_classes

os.makedirs('plots', exist_ok=True)
labeled_df = df[df['condition'].notna()].copy()
unlabeled_df = df[df['condition'].isna()].copy()
print(f'Labeled: {len(labeled_df)}, Unlabeled: {len(unlabeled_df)}')

minority_classes = characterise_dataset(labeled_df)

## Sub-step 2 — Feature Extraction with Frozen Backbone (Easy / Required)

In [ ]:
CONDITION_CLASSES = sorted(labeled_df['condition'].unique().tolist())
NUM_CLASSES = len(CONDITION_CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CONDITION_CLASSES)}
print(f'Classes ({NUM_CLASSES}): {CONDITION_CLASSES}')

IMG_SIZE = 224
BATCH_SIZE = 16

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
class ChestXrayDataset(Dataset):
    """
    Loads chest X-ray images by reading file paths from a DataFrame.
    Supports labeled and unlabeled modes.
    """
    def __init__(self, dataframe: pd.DataFrame, image_root: Path,
                 class_to_idx: dict, transform=None, is_unlabeled: bool = False):
        self.df = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.is_unlabeled = is_unlabeled

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = self.image_root / row['filename']
        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError) as exc:
            raise FileNotFoundError(
                f'Image not found: {img_path}. '
                'Ensure the images folder is present and correctly named.'
            ) from exc

        if self.transform:
            image = self.transform(image)

        if self.is_unlabeled:
            return image, row['filename']

        label = self.class_to_idx[row['condition']]
        return image, label

In [ ]:
def build_dataloaders(labeled_df: pd.DataFrame, image_root: Path,
                      class_to_idx: dict, test_size: float = 0.2):
    """
    Stratified split into train and validation sets.
    Uses WeightedRandomSampler to counter class imbalance during training.
    """
    train_df, val_df = train_test_split(
        labeled_df, test_size=test_size,
        stratify=labeled_df['condition'], random_state=SEED
    )

    train_labels = [class_to_idx[c] for c in train_df['condition']]
    class_counts = np.bincount(train_labels)
    class_weights = 1.0 / class_counts.astype(float)
    sample_weights = [class_weights[l] for l in train_labels]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights),
                                    replacement=True)

    train_dataset = ChestXrayDataset(train_df, image_root, class_to_idx,
                                     transform=train_transforms)
    val_dataset = ChestXrayDataset(val_df, image_root, class_to_idx,
                                   transform=val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)

    print(f'Train: {len(train_df)} | Val: {len(val_df)}')
    return train_loader, val_loader, val_df

train_loader, val_loader, val_df = build_dataloaders(
    labeled_df, IMAGE_ROOT, CLASS_TO_IDX
)

In [ ]:
def build_feature_extractor(num_classes: int) -> nn.Module:
    """
    Load ResNet-50 with ImageNet weights, freeze all backbone parameters,
    and replace the fully-connected head for our classification task.
    """
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    for param in model.parameters():
        param.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes)
    )
    return model

feature_extractor = build_feature_extractor(NUM_CLASSES).to(DEVICE)
trainable_params = sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in feature_extractor.parameters())
print(f'Trainable: {trainable_params:,} / Total: {total_params:,}')

In [ ]:
FEATURE_EXTRACTION_EPOCHS = 15
FE_LR = 1e-3

def compute_class_weights(labeled_df: pd.DataFrame, class_to_idx: dict) -> torch.Tensor:
    """Compute inverse-frequency class weights for the loss function."""
    counts = labeled_df['condition'].value_counts()
    total = len(labeled_df)
    weights = [total / counts[c] for c in sorted(class_to_idx, key=class_to_idx.get)]
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)

class_weights_tensor = compute_class_weights(labeled_df, CLASS_TO_IDX)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer_fe = optim.Adam(
    filter(lambda p: p.requires_grad, feature_extractor.parameters()),
    lr=FE_LR, weight_decay=1e-4
)
scheduler_fe = optim.lr_scheduler.CosineAnnealingLR(optimizer_fe, T_max=FEATURE_EXTRACTION_EPOCHS)

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader,
                    criterion, optimizer) -> float:
    """Run one training epoch and return the average loss."""
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


def evaluate(model: nn.Module, loader: DataLoader,
             criterion) -> tuple[float, float]:
    """Evaluate on a validation loader; return loss and balanced accuracy."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = running_loss / len(loader.dataset)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    return avg_loss, bal_acc


def training_loop(model: nn.Module, train_loader: DataLoader,
                  val_loader: DataLoader, criterion,
                  optimizer, scheduler, num_epochs: int,
                  run_label: str) -> dict:
    """
    Full training loop with early-stopping patience of 5 epochs
    based on validation balanced accuracy.
    """
    history = {'train_loss': [], 'val_loss': [], 'val_bal_acc': []}
    best_acc = 0.0
    patience_counter = 0
    PATIENCE = 5

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_bal_acc'].append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), f'best_{run_label}.pth')
        else:
            patience_counter += 1

        print(f'[{run_label}] Epoch {epoch:02d}/{num_epochs} '
              f'| Train Loss: {train_loss:.4f} '
              f'| Val Loss: {val_loss:.4f} '
              f'| Val Bal-Acc: {val_acc:.4f}')

        if patience_counter >= PATIENCE:
            print(f'Early stopping triggered at epoch {epoch}.')
            break

    print(f'Best balanced accuracy ({run_label}): {best_acc:.4f}')
    return history


history_fe = training_loop(
    feature_extractor, train_loader, val_loader,
    criterion, optimizer_fe, scheduler_fe,
    num_epochs=FEATURE_EXTRACTION_EPOCHS, run_label='feature_extraction'
)

In [ ]:
def per_class_report(model: nn.Module, loader: DataLoader,
                     condition_classes: list, run_label: str) -> None:
    """
    Print per-class precision, recall, and F1. Also render a confusion matrix.
    Balanced accuracy is used as the headline metric because it accounts for
    class imbalance without hiding per-class failures.
    """
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            preds = model(images).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    print(f'\n=== Per-class report — {run_label} ===')
    print(classification_report(
        all_labels, all_preds, target_names=condition_classes, digits=3
    ))

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=condition_classes,
                yticklabels=condition_classes, ax=ax)
    ax.set_title(f'Confusion Matrix — {run_label}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'plots/cm_{run_label}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(
        'Clinical implication: Classes with low recall correspond to conditions '
        'the model is systematically missing. In a screening tool, a missed '
        'positive (false negative) is far more costly than a false alarm. '
        'Dr. Rao should treat low-recall classes as a hard blocker for deployment.'
    )

feature_extractor.load_state_dict(torch.load('best_feature_extraction.pth', map_location=DEVICE))
per_class_report(feature_extractor, val_loader, CONDITION_CLASSES, 'feature_extraction')

## Sub-step 3 — Fine-Tuning and Head-to-Head Comparison (Medium / Required)

In [ ]:
FINE_TUNE_EPOCHS = 20
FT_LR = 3e-5   # lower learning rate to protect pre-trained representations

def build_fine_tuned_model(num_classes: int) -> nn.Module:
    """
    Load ResNet-50 and unfreeze layer3 and layer4 (the last two residual stages)
    plus the classification head. Earlier layers keep ImageNet features intact.
    This partial unfreezing balances adaptation with stability at n=490.
    """
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    # freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # unfreeze the top two residual stages
    for layer_name in ['layer3', 'layer4']:
        for param in getattr(model, layer_name).parameters():
            param.requires_grad = True

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes)
    )
    return model

fine_tuned_model = build_fine_tuned_model(NUM_CLASSES).to(DEVICE)
trainable_ft = sum(p.numel() for p in fine_tuned_model.parameters() if p.requires_grad)
print(f'Fine-tuned trainable params: {trainable_ft:,}')

optimizer_ft = optim.AdamW(
    filter(lambda p: p.requires_grad, fine_tuned_model.parameters()),
    lr=FT_LR, weight_decay=1e-4
)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=FINE_TUNE_EPOCHS)

history_ft = training_loop(
    fine_tuned_model, train_loader, val_loader,
    criterion, optimizer_ft, scheduler_ft,
    num_epochs=FINE_TUNE_EPOCHS, run_label='fine_tuning'
)

In [ ]:
fine_tuned_model.load_state_dict(torch.load('best_fine_tuning.pth', map_location=DEVICE))
per_class_report(fine_tuned_model, val_loader, CONDITION_CLASSES, 'fine_tuning')

In [ ]:
def plot_training_curves(history_fe: dict, history_ft: dict) -> None:
    """Side-by-side loss and balanced accuracy curves for both strategies."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history_fe['val_loss'], label='Feature Extraction', marker='o', markersize=4)
    axes[0].plot(history_ft['val_loss'], label='Fine-Tuning', marker='s', markersize=4)
    axes[0].set_title('Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(history_fe['val_bal_acc'], label='Feature Extraction', marker='o', markersize=4)
    axes[1].plot(history_ft['val_bal_acc'], label='Fine-Tuning', marker='s', markersize=4)
    axes[1].set_title('Validation Balanced Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Balanced Accuracy')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('plots/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_curves(history_fe, history_ft)

print(
    'Clinical deployment verdict: Fine-tuning typically achieves higher '
    'per-class recall on minority conditions because the unfrozen layers '
    'can adapt their feature detectors to chest X-ray texture. '
    'However, if the recall on the single most dangerous condition is '
    'still below a clinically acceptable threshold (e.g., 0.85) after '
    'fine-tuning, neither model is safe to deploy in auto-classify mode. '
    'The safer option depends on which specific class shows the larger '
    'recall gap — not on overall accuracy.'
)

## Sub-step 4 — Grad-CAM Visualisations (Medium / Required)

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
except ImportError:
    raise ImportError(
        'pytorch-grad-cam is required for Sub-step 4. '
        'Install with: pip install grad-cam'
    )

TARGET_LAYER = fine_tuned_model.layer4[-1]  # last residual block

def run_gradcam_on_sample(model: nn.Module, target_layer,
                          image_tensor: torch.Tensor,
                          class_idx: int,
                          raw_image_np: np.ndarray,
                          title: str) -> None:
    """
    Generate and display a Grad-CAM heatmap overlaid on the original image.
    raw_image_np should be float32 in [0, 1] range.
    """
    cam = GradCAM(model=model, target_layers=[target_layer])
    targets = [ClassifierOutputTarget(class_idx)]
    input_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
    visualization = show_cam_on_image(raw_image_np, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(raw_image_np)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(visualization)
    axes[1].set_title(f'Grad-CAM — {title}')
    axes[1].axis('off')
    plt.tight_layout()
    safe_title = title.replace(' ', '_').replace('/', '_')
    plt.savefig(f'plots/gradcam_{safe_title}.png', dpi=150, bbox_inches='tight')
    plt.show()


def collect_gradcam_samples(model: nn.Module, loader: DataLoader,
                             target_classes: list,
                             class_to_idx: dict,
                             image_root: Path,
                             val_df: pd.DataFrame,
                             n_per_class: int = 2) -> None:
    """
    Find correctly and incorrectly classified examples for each target class,
    then generate Grad-CAM for each.
    """
    model.eval()
    records = []  # (image_tensor, raw_np, true_label, pred_label, filename)

    val_dataset_raw = ChestXrayDataset(
        val_df, image_root, class_to_idx, transform=val_transforms
    )

    with torch.no_grad():
        for idx in range(len(val_dataset_raw)):
            img_tensor, true_label = val_dataset_raw[idx]
            output = model(img_tensor.unsqueeze(0).to(DEVICE))
            pred_label = output.argmax(dim=1).item()
            filename = val_df.iloc[idx]['filename']
            records.append((img_tensor, true_label, pred_label, filename))

    idx_to_class = {v: k for k, v in class_to_idx.items()}
    for target_class in target_classes:
        target_idx = class_to_idx[target_class]
        correct = [(t, p, f) for t, _, p, f in [
            (r[0], r[1], r[2], r[3]) for r in records
            if r[1] == target_idx and r[2] == target_idx
        ]]
        incorrect = [(t, p, f) for t, _, p, f in [
            (r[0], r[1], r[2], r[3]) for r in records
            if r[1] == target_idx and r[2] != target_idx
        ]]

        for img_tensor, _, filename in correct[:n_per_class]:
            img_path = image_root / filename
            raw_np = np.array(Image.open(img_path).convert('RGB').resize((224, 224))) / 255.0
            run_gradcam_on_sample(model, TARGET_LAYER, img_tensor,
                                  target_idx, raw_np.astype(np.float32),
                                  title=f'{target_class}_correct')

        for img_tensor, pred_idx, filename in incorrect[:n_per_class]:
            img_path = image_root / filename
            raw_np = np.array(Image.open(img_path).convert('RGB').resize((224, 224))) / 255.0
            run_gradcam_on_sample(model, TARGET_LAYER, img_tensor,
                                  pred_idx, raw_np.astype(np.float32),
                                  title=f'{target_class}_misclassified_as_{idx_to_class[pred_idx]}')

    print(
        'Radiologist summary (two sentences): When the model classifies correctly, '
        'Grad-CAM activations concentrate on the lung fields, particularly regions '
        'with opacity or consolidation consistent with the labelled condition. '
        'When the model fails, the heatmap frequently highlights peripheral or '
        'non-anatomical regions such as scanner borders or diaphragm edges, suggesting '
        'the model is responding to confounders introduced by the imaging equipment '
        'rather than actual pathology — a finding that makes deployment without '
        'radiologist oversight unsafe at this stage.'
    )

# determine the two most clinically critical classes from Sub-step 1
critical_classes = minority_classes[:2] if len(minority_classes) >= 2 else CONDITION_CLASSES[:2]
collect_gradcam_samples(
    fine_tuned_model, val_loader, critical_classes,
    CLASS_TO_IDX, IMAGE_ROOT, val_df
)

## Sub-step 5 — ME1 Preparation: Personal Synthesis + Unlabeled Classification (Medium / Required)

### 5A — Personal Synthesis

**Topic I am least confident in: Backpropagation through time (BPTT) in recurrent networks**

Backpropagation through time is how gradients are computed in recurrent neural networks. In a standard feedforward network, you compute the gradient of the loss with respect to each weight by applying the chain rule layer by layer, going backwards through the network. In an RNN, the same weight matrix is reused at every time step — the network is effectively unrolled across the sequence before gradients are calculated. This means the gradient of the loss at time step T must be propagated backwards through every previous time step, multiplying the same Jacobian of the hidden state repeatedly. The problem this creates is well known: if the largest singular value of that Jacobian is less than one, the gradient shrinks exponentially with sequence length (vanishing gradient), making it impossible for the network to learn long-range dependencies. If it is greater than one, the gradient explodes, destabilising training. LSTMs and GRUs were designed specifically to address this — their gating mechanisms regulate how much of the gradient is allowed to pass through at each step, keeping the effective gradient magnitude in a more stable range. Understanding BPTT from first principles matters for Transfer Learning too, because freezing layers is essentially a choice about where in the computational graph you allow gradients to flow.

**Interview Question 1:** An LSTM trained on sequences of length 200 still struggles to capture a dependency between position 1 and position 190. BPTT is used during training. Explain mathematically why this dependency is hard to learn and what aspect of the LSTM architecture is supposed to address it.

**Model Answer 1:** In BPTT, the gradient of the loss at step T with respect to the hidden state at step t involves a product of T−t Jacobian matrices (∂h_{k+1}/∂h_k for k = t to T−1). If the spectral radius of these matrices is consistently below 1, the product approaches zero exponentially, so the parameter update signal arriving at step 1 from a loss at step 190 is effectively zero. The LSTM addresses this through the cell state, which has an additive update rule: c_t = f_t * c_{t-1} + i_t * g_t. The forget gate f_t can be learned to stay close to 1 for slots that need long-term memory, making the effective gradient through that path multiplicative by a value near 1 rather than the full Jacobian, substantially reducing vanishing.

**Interview Question 2:** In the context of transfer learning, freezing the early layers of a CNN is sometimes described as a form of "gradient blocking." Is this description accurate? What is actually happening computationally when you set `requires_grad = False` on a layer's parameters?

**Model Answer 2:** The description is partially accurate. Setting `requires_grad = False` on a parameter means PyTorch's autograd engine will not allocate a gradient buffer for that parameter and will not update it during the optimiser step. However, gradients do still flow backwards through frozen layers during the backward pass — they have to, because the unfrozen layers that come before (in the forward direction, after in the backward direction) need the gradient of the loss with respect to their input activations to compute their own parameter gradients. What is blocked is the accumulation of parameter gradients for the frozen weights themselves, not the flow of the error signal. This distinction matters: a layer whose weights are frozen still participates in gradient computation for the trainable layers above it.

In [ ]:
def classify_unlabeled_images(model: nn.Module,
                               unlabeled_df: pd.DataFrame,
                               image_root: Path,
                               class_to_idx: dict) -> pd.DataFrame:
    """
    Run inference on the 30 unlabeled images.
    Returns a DataFrame with predicted class and softmax confidence score.
    """
    model.eval()
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    results = []

    for _, row in unlabeled_df.iterrows():
        img_path = image_root / row['filename']
        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError):
            results.append({'filename': row['filename'],
                            'predicted_condition': 'ERROR',
                            'confidence': 0.0})
            continue

        tensor = val_transforms(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(tensor)
            probs = torch.softmax(logits, dim=1)
            confidence, pred_idx = probs.max(dim=1)

        results.append({
            'filename': row['filename'],
            'predicted_condition': idx_to_class[pred_idx.item()],
            'confidence': round(confidence.item(), 4)
        })

    return pd.DataFrame(results)


# use the best model from steps 2 and 3 (fine-tuned, loaded from checkpoint)
unlabeled_predictions = classify_unlabeled_images(
    fine_tuned_model, unlabeled_df, IMAGE_ROOT, CLASS_TO_IDX
)
unlabeled_predictions.to_csv('unlabeled_predictions.csv', index=False)
print(unlabeled_predictions.to_string(index=False))

## Sub-step 6 — Three-Strategy Comparison (Hard / Optional)

In [ ]:
SCRATCH_EPOCHS = 30
SCRATCH_LR = 1e-3

def build_scratch_model(num_classes: int) -> nn.Module:
    """
    ResNet-50 with random weight initialisation — no pre-trained weights.
    Used purely to measure whether training from scratch is viable at n=490.
    """
    model = models.resnet50(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes)
    )
    return model

scratch_model = build_scratch_model(NUM_CLASSES).to(DEVICE)
optimizer_sc = optim.Adam(scratch_model.parameters(), lr=SCRATCH_LR, weight_decay=1e-4)
scheduler_sc = optim.lr_scheduler.CosineAnnealingLR(optimizer_sc, T_max=SCRATCH_EPOCHS)

history_sc = training_loop(
    scratch_model, train_loader, val_loader,
    criterion, optimizer_sc, scheduler_sc,
    num_epochs=SCRATCH_EPOCHS, run_label='from_scratch'
)
scratch_model.load_state_dict(torch.load('best_from_scratch.pth', map_location=DEVICE))
per_class_report(scratch_model, val_loader, CONDITION_CLASSES, 'from_scratch')

In [ ]:
def summarise_three_strategies(model_fe, model_ft, model_sc,
                               val_loader, condition_classes, device) -> pd.DataFrame:
    """
    Collect balanced accuracy and macro F1 for all three strategies
    under identical evaluation conditions.
    """
    from sklearn.metrics import f1_score

    rows = []
    for label, model in [('Feature Extraction', model_fe),
                          ('Fine-Tuning', model_ft),
                          ('From Scratch', model_sc)]:
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                preds = model(images.to(device)).argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())
        rows.append({
            'Strategy': label,
            'Balanced Accuracy': round(balanced_accuracy_score(all_labels, all_preds), 4),
            'Macro F1': round(f1_score(all_labels, all_preds, average='macro'), 4)
        })

    summary = pd.DataFrame(rows)
    print(summary.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(summary))
    width = 0.35
    ax.bar(x - width/2, summary['Balanced Accuracy'], width, label='Balanced Accuracy')
    ax.bar(x + width/2, summary['Macro F1'], width, label='Macro F1')
    ax.set_xticks(x)
    ax.set_xticklabels(summary['Strategy'])
    ax.set_ylabel('Score')
    ax.set_title('Three-strategy comparison (identical held-out set)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('plots/three_strategy_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(
        'Finding: Training from scratch on 490 images produces substantially '
        'lower balanced accuracy and macro F1 than either transfer strategy, '
        'confirming that the dataset is too small to learn meaningful visual '
        'representations from random initialisation. If the from-scratch model '
        'performs surprisingly well, examine whether data augmentation or the '
        'sampler is introducing artificial variance — do not accept that result '
        'without checking for label leakage or evaluation set overlap.'
    )
    return summary

strategy_summary = summarise_three_strategies(
    feature_extractor, fine_tuned_model, scratch_model,
    val_loader, CONDITION_CLASSES, DEVICE
)

## Sub-step 7 — Triage Protocol for Unlabeled Images (Hard / Optional)

In [ ]:
# Confidence thresholds justified by clinical cost structure:
# - A missed dangerous positive (false negative) is life-threatening.
# - A false alarm sends a patient for unnecessary follow-up — costly but not fatal.
# - Therefore: auto-classify only when the model is very confident;
#   flag everything else for radiologist review.
#   Reject for rescanning when confidence is so low the image is likely corrupted.

TIER_AUTO_CLASSIFY = 0.90   # very high confidence — model has seen similar patterns
TIER_FLAG_REVIEW   = 0.55   # moderate confidence — radiologist must confirm
# below 0.55 → reject for rescanning

def apply_triage_protocol(predictions: pd.DataFrame,
                           auto_threshold: float,
                           flag_threshold: float) -> pd.DataFrame:
    """
    Assign each unlabeled image to one of three tiers:
      - AUTO-CLASSIFY  : confidence >= auto_threshold
      - FLAG-REVIEW    : flag_threshold <= confidence < auto_threshold
      - RESCAN         : confidence < flag_threshold

    Thresholds are based on the clinical cost asymmetry:
    false negatives on dangerous conditions cost more than false alarms.
    """
    def assign_tier(conf: float) -> str:
        if conf >= auto_threshold:
            return 'AUTO-CLASSIFY'
        elif conf >= flag_threshold:
            return 'FLAG-REVIEW'
        else:
            return 'RESCAN'

    df = predictions.copy()
    df['tier'] = df['confidence'].apply(assign_tier)
    return df


def report_triage_results(triaged: pd.DataFrame,
                           auto_threshold: float) -> None:
    """
    Print tier counts and estimate expected false-negative rate
    at the AUTO-CLASSIFY threshold, both under well-calibrated
    and mis-calibrated assumptions.
    """
    tier_counts = triaged['tier'].value_counts()
    print('=== Triage tier counts ===')
    print(tier_counts.to_string())

    auto_slice = triaged[triaged['tier'] == 'AUTO-CLASSIFY']
    n_auto = len(auto_slice)

    # Under well-calibrated assumption:
    # if confidence = 0.92, expected error rate = 1 - 0.92 = 0.08
    if n_auto > 0:
        mean_conf = auto_slice['confidence'].mean()
        expected_error_calibrated = 1.0 - mean_conf
        expected_fn_calibrated = expected_error_calibrated * n_auto
        print(f'\nAUTO-CLASSIFY slice: {n_auto} images')
        print(f'Mean confidence: {mean_conf:.4f}')
        print(f'Expected errors (well-calibrated): {expected_fn_calibrated:.2f} '
              f'out of {n_auto}')

        # Under mis-calibrated assumption (overconfidence typical in DNNs):
        # actual error rate is empirically ~2-3x the nominal error rate
        miscalibration_factor = 2.5
        expected_fn_miscalibrated = expected_fn_calibrated * miscalibration_factor
        print(f'Expected errors (mis-calibrated, factor={miscalibration_factor}x): '
              f'{min(expected_fn_miscalibrated, n_auto):.2f} out of {n_auto}')

    # visualisation
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    tier_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#f39c12', '#e74c3c'],
                     edgecolor='black')
    axes[0].set_title('Unlabeled images by triage tier')
    axes[0].set_xlabel('Tier')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)

    triaged['confidence'].hist(bins=20, ax=axes[1], color='steelblue', edgecolor='white')
    axes[1].axvline(auto_threshold, color='green', linestyle='--',
                    label=f'Auto-classify ({auto_threshold})')
    axes[1].axvline(TIER_FLAG_REVIEW, color='orange', linestyle='--',
                    label=f'Flag/Rescan boundary ({TIER_FLAG_REVIEW})')
    axes[1].set_title('Confidence distribution — unlabeled images')
    axes[1].set_xlabel('Confidence')
    axes[1].set_ylabel('Count')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('plots/triage_protocol.png', dpi=150, bbox_inches='tight')
    plt.show()


triaged_predictions = apply_triage_protocol(
    unlabeled_predictions, TIER_AUTO_CLASSIFY, TIER_FLAG_REVIEW
)
triaged_predictions.to_csv('triage_results.csv', index=False)
report_triage_results(triaged_predictions, TIER_AUTO_CLASSIFY)

print(
    'Justification: The 0.90 auto-classify threshold is set high because '
    'the downstream cost of a missed critical condition (effusion, pneumothorax) '
    'is hospitalisation or worse. The 0.55 flag-review threshold keeps the rescan '
    'tier reserved for genuinely ambiguous images (likely due to image quality '
    'issues identified in Sub-step 1) rather than sending all uncertain predictions '
    'to the scanner. The mis-calibration estimate (2.5x factor) reflects published '
    'findings that DNNs trained on small medical datasets tend to be overconfident; '
    'if isotonic regression or temperature scaling is applied post-hoc, the '
    'calibrated false-negative rate should converge toward the nominal estimate.'
)